# 🎮 PokemonAgent Training on Google Colab

Train your MaskablePPO Pokémon agent directly on Google Colab with a native Pokémon Showdown server!

## Cell 1️⃣: Install Node.js

In [ ]:
# Install Node.js (required for Pokémon Showdown server)
!curl -fsSL https://deb.nodesource.com/setup_18.x | sudo -E bash -
!apt-get install -y nodejs
!node --version
!npm --version
print("✅ Node.js installed!")

## Cell 2️⃣: Clone Repository & Install Python Dependencies

In [ ]:
import os

# Clone the repository (UPDATE THIS WITH YOUR REPO URL)
repo_url = "https://github.com/your-username/PokemonAgent.git"
!cd /content && git clone {repo_url}
os.chdir('/content/PokemonAgent')

# Install Python dependencies
!pip install -q -r requirements.txt

print("✅ Repository and dependencies installed!")

## Cell 3️⃣: Build Pokémon Showdown Server

In [ ]:
import os

# Clone Pokémon Showdown
print("⏳ Cloning Pokémon Showdown...")
!cd /content && git clone --depth 1 https://github.com/smogon/pokemon-showdown.git
os.chdir('/content/pokemon-showdown')

# Build the server
print("⏳ Installing dependencies and building...")
!npm install -q
!node build

print("✅ Pokémon Showdown server built successfully!")

## Cell 4️⃣: Start Pokémon Showdown Server (Background)

In [ ]:
import subprocess
import time
import os

os.chdir('/content/pokemon-showdown')

# Start the server in the background
print("🚀 Starting Pokémon Showdown server on localhost:8000...")
server_process = subprocess.Popen(
    ["node", "pokemon-showdown", "start", "--no-security"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Give server time to start
time.sleep(5)

# Check if server is running
if server_process.poll() is None:
    print(f"✅ Server started successfully!")
    print(f"   Process ID: {server_process.pid}")
else:
    stdout, stderr = server_process.communicate()
    print(f"❌ Server failed to start")
    print(f"Error: {stderr}")

## Cell 5️⃣: Verify Server Connection

In [ ]:
import urllib.request
import urllib.error
import time

def test_server():
    """Test if the Pokémon Showdown server is reachable."""
    max_retries = 15
    for attempt in range(max_retries):
        try:
            response = urllib.request.urlopen("http://localhost:8000/", timeout=2)
            print(f"✅ Server is running and reachable!")
            return True
        except (urllib.error.URLError, ConnectionRefusedError) as e:
            if attempt < max_retries - 1:
                print(f"⏳ Waiting for server (attempt {attempt + 1}/{max_retries})...")
                time.sleep(2)
            else:
                print(f"❌ Cannot reach server: {e}")
                return False
    return False

test_server()

## Cell 6️⃣: Configure W&B (Optional but Recommended)

Get your API key from https://wandb.ai/settings/api

In [ ]:
import os

# Set your W&B API key (optional, but recommended for tracking)
# Get it from: https://wandb.ai/settings/api
wandb_api_key = "your-wandb-api-key-here"

if wandb_api_key != "your-wandb-api-key-here":
    os.environ["WANDB_API_KEY"] = wandb_api_key
    print("✅ W&B configured")
else:
    print("⚠️  W&B API key not set (optional)")

---
## 🎓 Supervised Pretraining (Optional but Recommended)

Run these cells **before** RL training to bootstrap the policy from human Pokémon Showdown replays.
This uses behavioral cloning + value pretraining so the agent starts from human-like play instead of random moves.

**Pipeline:** Download replays → Convert to observations → Pretrain → RL fine-tune

> Skip to **Cell 7** if you want to train from scratch (random policy).

## Cell S1️⃣: Install Supervised Pretraining Dependencies

In [ ]:
# Extra dependencies for downloading HuggingFace replay data and decompressing lz4 files
!pip install -q huggingface_hub lz4
print("✅ Supervised pretraining dependencies installed!")

## Cell S2️⃣: Download Replay Data from HuggingFace

Downloads Gen 1 OU replays from [jakegrigsby/metamon-parsed-replays](https://huggingface.co/datasets/jakegrigsby/metamon-parsed-replays).  
**Size:** ~few hundred MB. Change `formats` below to include more tiers.

In [ ]:
import os
os.chdir('/content/PokemonAgent')

# Formats to download — add 'gen1uu', 'gen1nu', 'gen1ubers' for more data
formats = ["gen1ou"]

for fmt in formats:
    print(f"⏳ Downloading {fmt} replays...")
    !python data_generation/download_metamon_hf.py \
        --format {fmt} \
        --output-dir data/metamon

print("✅ Replay data downloaded to data/metamon/")

## Cell S3️⃣: Convert Replays to Observation Arrays

Reads every `.json.lz4` battle file, runs each timestep through `BattleStateGen1.to_array()`,
and writes one JSON record per timestep to a `.jsonl` file.

> **Quick test:** Set `max_battles = 100` to convert a small subset first.

In [ ]:
import os
os.chdir('/content/PokemonAgent')

# Set to an integer (e.g. 500) for a faster smoke-test, or None to convert everything
max_battles = None

os.makedirs('data/supervised', exist_ok=True)

max_battles_flag = f"--max-battles {max_battles}" if max_battles else ""

print("⏳ Converting replays to observation arrays...")
!python data_generation/convert_metamon_json.py \
    --input-dir data/metamon/gen1ou \
    --output    data/supervised/gen1ou.jsonl \
    {max_battles_flag}

# Show how many timesteps were written
import subprocess
result = subprocess.run(['wc', '-l', 'data/supervised/gen1ou.jsonl'], capture_output=True, text=True)
print(f"✅ Conversion complete! {result.stdout.strip()} timesteps written")

## Cell S4️⃣: Run Supervised Pretraining

Trains the policy via behavioral cloning (imitate human moves) + value prediction (predict win/loss).  
Loss: `L = λ_value · MSE(V(s), Gₜ) + λ_bc · CrossEntropy(π(s), a_human)`

Expected validation accuracy: **35–50%** is normal for Gen 1 OU (sufficient to warm-start RL).

In [ ]:
import os
os.chdir('/content/PokemonAgent')

os.makedirs('models', exist_ok=True)

# Adjust epochs and batch-size to taste; reduce batch-size if you hit OOM
!python -m training.supervised.run_supervised \
    --data-path   data/supervised/gen1ou.jsonl \
    --epochs      20 \
    --batch-size  256 \
    --val-split   0.1 \
    --device      cuda \
    --output-path models/supervised_gen1_6v6.zip

print("✅ Supervised pretraining complete! Checkpoint saved to models/supervised_gen1_6v6.zip")

---
## Cell 7️⃣: 🎯 RUN RL TRAINING

**Option A (Recommended):** Fine-tune from the supervised checkpoint (run Cells S1–S4 first).  
**Option B:** Train from scratch with a random policy.

In [ ]:
import os
os.chdir('/content/PokemonAgent')

# ─── Option A: Fine-tune from supervised checkpoint (recommended) ───────────
# Uncomment if you completed the supervised pretraining cells above:
# !python -m training.train \
#   --load-model-path models/supervised_gen1_6v6 \
#   --train-team steelix \
#   --pool-all \
#   --timesteps 500000 \
#   --rounds-per-opponent 2000 \
#   --device cuda \
#   --seed 42

# ─── Option B: Train from scratch (random policy) ──────────────────────────
!python -m training.train \
  --train-team steelix \
  --pool-all \
  --timesteps 100000 \
  --rounds-per-opponent 2000 \
  --device cuda \
  --seed 42

## Cell 8️⃣ (Optional): Download Trained Model

In [ ]:
import os
from google.colab import files

# Compress the trained model
os.system("cd /content/PokemonAgent && zip -r /tmp/trained_model.zip data/1v1*")

# Download to your computer
files.download("/tmp/trained_model.zip")
print("✅ Model downloaded to your computer!")

## Cell 9️⃣ (Optional): Stop Server & Cleanup

In [ ]:
# Kill the Pokémon Showdown server process
import subprocess

# Find and kill the pokemon-showdown process
result = subprocess.run(['pgrep', '-f', 'pokemon-showdown'], capture_output=True, text=True)
if result.stdout.strip():
    pids = result.stdout.strip().split('\n')
    for pid in pids:
        print(f"🛑 Killing process {pid}...")
        os.kill(int(pid), 9)
    print("✅ Server stopped")
else:
    print("ℹ️  No server process found (already stopped)")

## 📋 Training Command Presets

Replace the command in Cell 7 with one of these presets.

### Quick Test — RL from scratch (5 min)
```bash
python -m training.train \
  --train-team steelix \
  --pool toxapex \
  --timesteps 10000 \
  --rounds-per-opponent 100 \
  --device cuda
```

### Balanced Training — RL from scratch (1-2 hours)
```bash
python -m training.train \
  --train-team garchomp \
  --pool-all \
  --timesteps 200000 \
  --rounds-per-opponent 2000 \
  --device cuda
```

### Full Training with Evaluation — RL from scratch (3-5 hours)
```bash
python -m training.train \
  --train-team steelix \
  --pool-all \
  --timesteps 500000 \
  --rounds-per-opponent 2000 \
  --eval-every-timesteps 50000 \
  --eval-episodes 100 \
  --device cuda
```

### Fine-tune from Supervised Checkpoint (recommended — run Cells S1-S4 first)
```bash
python -m training.train \
  --load-model-path models/supervised_gen1_6v6 \
  --train-team steelix \
  --pool-all \
  --timesteps 500000 \
  --rounds-per-opponent 2000 \
  --device cuda
```

### Inline Supervised Warmup + RL (no separate pretraining step)
```bash
python -m training.train \
  --warmup-data-path data/supervised/gen1ou.jsonl \
  --warmup-epochs    20 \
  --train-team steelix \
  --pool-all \
  --timesteps 500000 \
  --device cuda
```